In [1]:
import torch
import gc

# Delete the model and processor variables if they exist
try:
    del model
    del processor
except NameError:
    pass

# Force Python's garbage collector to run
gc.collect()

# Empty the PyTorch CUDA cache
torch.cuda.empty_cache()

In [2]:
import os

# 1. Route all Hugging Face downloads to your spacious persistent volume
os.environ["HF_HOME"] = "/workspace/huggingface_cache"

# 2. Keep the fix for the Xet network timeout issue
os.environ["HF_HUB_DISABLE_XET"] = "1"

print(f"Hugging Face cache set to: {os.environ['HF_HOME']}")

Hugging Face cache set to: /workspace/huggingface_cache


In [4]:
# !pip install -U torch torchvision torchaudio
# !pip install -U transformers accelerate peft bitsandbytes trl datasets qwen-vl-utils
!pip install -U transformers accelerate peft bitsandbytes trl datasets qwen-vl-utils

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 111.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 91.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 88.4 MB/s eta 0:00:00a 0:00:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.0/751.0 kB 136.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 103.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 661.5/661.5 kB 117.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.8/48.8 MB 127.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 799.8/799.8 kB 138.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 116.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 96.8 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 132.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 108.0 MB/s eta 0:00:00
   ━━━━━━━━━

In [6]:
!pip uninstall -y transformers
!pip install git+https://github.com/huggingface/transformers.git

Found existing installation: transformers 5.8.0
Uninstalling transformers-5.8.0:
  Successfully uninstalled transformers-5.8.0
  Cloning https://github.com/huggingface/transformers.git to /tmp/pip-req-build-_j6831r9
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers.git /tmp/pip-req-build-_j6831r9
  Resolved https://github.com/huggingface/transformers.git to commit c9de1097eed992fe205f876415e7a7cfaa06be50
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for transformers: filename=transformers-5.8.0.dev0-py3-none-any.whl size=11875973 sha256=7b08cfe4e27081d62f1d166b41ad0599d24ff4097a923e58efee958c12341378
  Stored in directory: /tmp/pip-ephem-wheel-cache-h84qdza8/wheels/32/4b/78/f195c684dd3a9ed21f3b39fe8f85b48df7918581b6437be143
Successfully built transformers

[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, 

In [3]:
import json
import os
from datasets import Dataset

WORKSPACE_DIR = "/workspace/"
LOCAL_IMAGE_DIR = os.path.join(WORKSPACE_DIR, "final_balanced_images")

def load_and_fix_paths(json_path):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
        
    processed_data = []
    for item in data:
        raw_img_path = item["images"][0].replace("\\", "/")
        img_filename = os.path.basename(raw_img_path)
        local_img_path = os.path.join(LOCAL_IMAGE_DIR, img_filename)
        
        processed_data.append({
            "id": str(item["id"]),  # <--- THE FIX: Cast all IDs to strings
            "messages": item["messages"],
            "images": [local_img_path]
        })
    return Dataset.from_list(processed_data)

# Load Train and Validation sets
train_dataset = load_and_fix_paths(os.path.join(WORKSPACE_DIR, "qwen25_vl_3b_train.json"))
val_dataset = load_and_fix_paths(os.path.join(WORKSPACE_DIR, "qwen25_vl_3b_val.json"))

print(f"Loaded {len(train_dataset)} Train samples and {len(val_dataset)} Val samples.")

Loaded 2010 Train samples and 251 Val samples.


In [4]:
import torch
from qwen_vl_utils import process_vision_info

class QwenDataCollator:
    def __init__(self, processor):
        self.processor = processor

    def __call__(self, examples):
        batch_messages = []
        
        for example in examples:
            raw_messages = example["messages"]
            img_path = example["images"][0] 
            
            formatted_messages = []
            image_added = False # Track if we've already added the image
            
            for msg in raw_messages:
                if msg["role"] == "user":
                    clean_text = msg["content"].replace("<|image_pad|>", "").strip()
                    
                    # Only append the image to the very first user prompt
                    if not image_added:
                        formatted_messages.append({
                            "role": "user",
                            "content": [
                                {"type": "image", "image": img_path,
                                    "max_pixels": 512*512},
                                {"type": "text", "text": clean_text}
                            ]
                        })
                        image_added = True
                    else:
                        formatted_messages.append({
                            "role": "user",
                            "content": clean_text
                        })
                else:
                    formatted_messages.append(msg)
            
            batch_messages.append(formatted_messages)

        # Apply chat template
        texts = [
            self.processor.apply_chat_template(msg, tokenize=False, add_generation_prompt=False) 
            for msg in batch_messages
        ]
        
        image_inputs, video_inputs = process_vision_info(batch_messages)

        batch = self.processor(
            text=texts,
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt"
        )

        # ---> THE SILVER BULLET FIX <---
        # Force PyTorch autograd to track the vision tower graph during checkpointing
        if "pixel_values" in batch and batch["pixel_values"] is not None:
            batch["pixel_values"] = batch["pixel_values"].requires_grad_(True)
        if "pixel_values_videos" in batch and batch["pixel_values_videos"] is not None:
            batch["pixel_values_videos"] = batch["pixel_values_videos"].requires_grad_(True)

        # ---> SFT LABEL MASKING FIX <---
        labels = batch["input_ids"].clone()
        labels[labels == self.processor.tokenizer.pad_token_id] = -100
        
        im_start_id = self.processor.tokenizer.convert_tokens_to_ids("<|im_start|>")
        im_end_id = self.processor.tokenizer.convert_tokens_to_ids("<|im_end|>")
        
        for i in range(len(labels)):
            in_assistant_turn = False
            j = 0
            while j < len(labels[i]):
                token = labels[i][j]
                
                if token == im_start_id:
                    # Look ahead to see if the next token(s) decode to "assistant"
                    # We check the next 2 tokens just in case of weird tokenization splits
                    lookahead_tokens = labels[i][j+1:j+3].tolist()
                    lookahead_text = self.processor.tokenizer.decode(lookahead_tokens)
                    
                    if "assistant" in lookahead_text:
                        in_assistant_turn = True
                        labels[i][j] = -100  # Mask the <|im_start|> token itself
                        j += 1
                        continue
                
                if token == im_end_id:
                    in_assistant_turn = False
                    # We usually don't mask the <|im_end|> token so the model learns to stop
                    
                if not in_assistant_turn:
                    labels[i][j] = -100
                
                j += 1
                
        batch["labels"] = labels
        
        return batch

In [5]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig
from transformers.trainer_utils import get_last_checkpoint
import os

model_id = "Qwen/Qwen2.5-VL-3B-Instruct"
processor = AutoProcessor.from_pretrained(model_id)

# 1. Load the model directly in native bfloat16 (No BitsAndBytes!)
model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

# 2. Safely freeze the vision tower
for name, param in model.named_parameters():
    if "visual" in name:
        param.requires_grad = False

# 3. Enable gradient checkpointing for memory efficiency
model.gradient_checkpointing_enable()
model.config.use_cache = False

# 4. Inject LoRA adapters directly
lora_config = LoraConfig(
    r=64,
    lora_alpha=128,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj",
                    "o_proj", "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)

data_collator = QwenDataCollator(processor)

training_args = SFTConfig(
    output_dir="/workspace/qwen3b_lora_sft_bf16",

    # Memory Management
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    # effective batch size of 16
    gradient_accumulation_steps=16,
    max_length=3072,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    # Evaluation Settings
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",

    # Learning Parameters
    num_train_epochs=3,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,

    save_total_limit=3,
    logging_steps=10,

    bf16=True,                # <--- Disabled
    fp16=False,
    # <--- Switched from paged_adamw_8bit to standard torch AdamW
    optim="adamw_torch",
    remove_unused_columns=False,
    dataset_kwargs={"skip_prepare_dataset": True}
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=training_args,
    data_collator=data_collator,
    processing_class=processor,
)

last_checkpoint = None
if os.path.isdir(training_args.output_dir):
    last_checkpoint = get_last_checkpoint(training_args.output_dir)

if last_checkpoint is not None:
    print(f"Resuming training from checkpoint: {last_checkpoint}")
else:
    print("Starting new native bf16 training run...")

trainer.train(resume_from_checkpoint=last_checkpoint)
trainer.model.save_pretrained("/workspace/qwen3b_lora_sft_bf16_final")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 151645, 'bos_token_id': None, 'pad_token_id': 151643}.


Starting new native bf16 training run...


/usr/local/lib/python3.11/dist-packages/torch/utils/checkpoint.py:1399: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with device_autocast_ctx, torch.cpu.amp.autocast(**cpu_autocast_kwargs), recompute_context:  # type: ignore[attr-defined]


Step,Training Loss,Validation Loss
50,0.219044,0.193920
100,0.156198,0.159978
150,0.118682,0.142235
200,0.111822,0.125952
250,0.089106,0.111400
300,0.074330,0.105437
350,0.074475,0.101539
378,0.078724,0.101455


/usr/local/lib/python3.11/dist-packages/torch/utils/checkpoint.py:1399: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with device_autocast_ctx, torch.cpu.amp.autocast(**cpu_autocast_kwargs), recompute_context:  # type: ignore[attr-defined]
/usr/local/lib/python3.11/dist-packages/torch/utils/checkpoint.py:1399: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with device_autocast_ctx, torch.cpu.amp.autocast(**cpu_autocast_kwargs), recompute_context:  # type: ignore[attr-defined]
/usr/local/lib/python3.11/dist-packages/torch/utils/checkpoint.py:1399: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with device_autocast_ctx, torch.cpu.amp.autocast(**cpu_autocast_kwargs), recompute_context:  # type: ignore[attr-defined]
/usr/local/lib/python3.11/dist-packages/torch/uti